### Step 5 - Evaluation

Measures the fine-tuned model against the base model on the 51 test traces, and probes two
dataset leakages.

The four conditions:

| | what changes in the prompt | what it tests |
|---|---|---|
| **A** | nothing | the fine-tuned model's performance |
| **B** | tool order shuffled | does the model use the correct tool's position? |
| **C** | without the LoRA adapter | how much the training bought |
| **D** | tool count inverted | does the model count how many tools are listed? |

#### 1. Load the data

In [1]:
import json, re, collections

with open("dados/teste.jsonl", encoding="utf-8") as f:
    testes = [json.loads(linha) for linha in f]

saidas = {
    "A original":    json.load(open("resultados/saidas_ft.json",   encoding="utf-8")),
    "B embaralhada": json.load(open("resultados/saidas_shuf.json", encoding="utf-8")),
    "C base":        json.load(open("resultados/saidas_base.json", encoding="utf-8")),
}

tools_catalogo = json.load(open("dados/tools_catalogo.json", encoding="utf-8"))
indice_tools = {t["nome"]: t for t in tools_catalogo}

com_tool = [t for t in testes if len(t["messages"]) == 5]
sem_tool = [t for t in testes if len(t["messages"]) == 3]

print(f"{len(testes)} testes  |  {len(com_tool)} com tool  |  {len(sem_tool)} sem tool")
print({nome: len(s) for nome, s in saidas.items()})

51 testes  |  37 com tool  |  14 sem tool
{'A original': 51, 'B embaralhada': 51, 'C base': 51}


##### Splits the model's raw output into `(type, content)`

In [2]:
def extrair(saida):
    """Devolve (tipo, conteudo) a partir da saída bruta do modelo."""
    tool  = re.search(r"<tool_call>\s*(.*?)\s*</tool_call>",       saida, re.DOTALL)
    final = re.search(r"<final_answer>\s*(.*?)\s*</final_answer>", saida, re.DOTALL)

    if tool and final:
        return "ambos", saida
    if tool:
        return "tool_call", tool.group(1)
    if final:
        return "final_answer", final.group(1)
    return "sem_tag", saida

In [3]:
# testa nos gabaritos do dataset (conhecidamente bons) e nos dois caminhos de falha
for t in [com_tool[0], sem_tool[0]]:
    tipo, conteudo = extrair(t["messages"][2]["content"])
    print(f"{tipo:14} | {conteudo[:60]}")

print(extrair("só um texto solto, sem tag nenhuma")[0])   # esperado: sem_tag
print(extrair('<tool_call>{"a": 1}')[0])                  # abriu e não fechou -> sem_tag

tool_call      | {"nome_tool": "create_budget", "argumentos": {"income": 5000
final_answer   | Melhorar as habilidades de resolução de conflitos requer prá
sem_tag
sem_tag


##### 2. The meter

Five tests **in cascade**, each one only makes sense if the previous one passed, and each
fails for a different reason:

| test | what happens if it fails | how to fix it |
|---|---|---|
| `decidiu_certo` | calls the API without needing to, or answers when it should have looked it up | more `sem_tool` examples |
| `json_valido` | the program **breaks** | more training |
| `tool_certa` | the API is called **successfully** and does the wrong thing | ambiguous descriptions |
| `params_ok` | the API rejects it, or silently ignores an unknown field | badly specified schema |
| `tipos_ok` | `"5000"` where `5000` was expected | same |

Failing `tool_certa` is **worse** than failing `json_valido`: invalid JSON breaks and you
see it; the wrong tool returns 200 and transfers money to the wrong account.

**Fields that do not apply stay `None`, never `False`.** In a `sem_tool` case there is no
tool to get right - marking it `False` would contaminate the average with 14 cases that
have no ground truth.

In [ ]:
MAPA_TIPOS = {"string": str, "number": (int, float), "integer": int,
              "boolean": bool, "array": list, "object": dict}


def avaliar(exemplo, saida):
    esperado = "tool_call" if len(exemplo["messages"]) == 5 else "final_answer"
    tipo, conteudo = extrair(saida)

    r = {"esperado": esperado, "obtido": tipo,
         "decidiu_certo": tipo == esperado,
         "json_valido": None, "tool_certa": None,
         "params_ok": None, "tipos_ok": None}

    if esperado != "tool_call" or tipo != "tool_call":
        return r

    try:
        chamada = json.loads(conteudo)
    except json.JSONDecodeError:
        r["json_valido"] = False
        return r
    r["json_valido"] = True

    _, gabarito = extrair(exemplo["messages"][2]["content"])
    nome_certo  = json.loads(gabarito)["nome_tool"]
    nome_obtido = chamada.get("nome_tool")
    r["tool_certa"] = nome_obtido == nome_certo

    schema = indice_tools.get(nome_obtido)
    if schema is None:                 # inventou uma tool que não existe
        r["params_ok"] = False
        return r

    args = chamada.get("argumentos")
    if not isinstance(args, dict):
        r["params_ok"] = False
        return r

    permitidos   = {p["nome"] for p in schema["parametros"]}
    obrigatorios = {p["nome"] for p in schema["parametros"] if p["obrigatorio"]}
    # encadeamento de conjuntos: todo obrigatório presente e nada presente é desconhecido
    r["params_ok"] = obrigatorios <= args.keys() <= permitidos

    r["tipos_ok"] = all(
        isinstance(args[p["nome"]], MAPA_TIPOS.get(p["tipo"], object))
        for p in schema["parametros"] if p["nome"] in args
    )
    return r

In [ ]:
CAMPOS = ["decidiu_certo", "json_valido", "tool_certa", "params_ok", "tipos_ok"]


def resumir(nome, saidas_da_condicao):
    res = [avaliar(ex, s) for ex, s in zip(testes, saidas_da_condicao)]
    print(f"\n### {nome}")
    for campo in CAMPOS:
        v = [r[campo] for r in res if r[campo] is not None]  
        print(f"  {campo:15} {f'{sum(v)}/{len(v)} = {sum(v)/len(v):.0%}' if v else '—'}")
    print("  obtido:", dict(collections.Counter(r["obtido"] for r in res)))
    return res

##### 3. Run the three conditions

The numbers have to match the ones measured on Colab: **A 51/51, B 49/51, C 27/51**, with
B's failures on examples 15, 26 and 36.

In [7]:
resultados = {nome: resumir(nome, s) for nome, s in saidas.items()}

print("\n=== falhas, com a pergunta ===")
for nome, res in resultados.items():
    for i, r in enumerate(res):
        ruins = [k for k in CAMPOS if r[k] is False]
        if ruins:
            print(f"[{nome}] {i:3} {ruins}")
            print(f"          {testes[i]['messages'][1]['content'][:75]}")


### A original
  decidiu_certo   51/51 = 100%
  json_valido     37/37 = 100%
  tool_certa      37/37 = 100%
  params_ok       37/37 = 100%
  tipos_ok        37/37 = 100%
  obtido: {'final_answer': 14, 'tool_call': 37}

### B embaralhada
  decidiu_certo   49/51 = 96%
  json_valido     34/35 = 97%
  tool_certa      34/34 = 100%
  params_ok       34/34 = 100%
  tipos_ok        34/34 = 100%
  obtido: {'final_answer': 14, 'tool_call': 35, 'sem_tag': 2}

### C base
  decidiu_certo   27/51 = 53%
  json_valido     17/17 = 100%
  tool_certa      17/17 = 100%
  params_ok       17/17 = 100%
  tipos_ok        17/17 = 100%
  obtido: {'sem_tag': 21, 'final_answer': 12, 'tool_call': 17, 'ambos': 1}

=== falhas, com a pergunta ===
[B embaralhada]  15 ['json_valido']
          Eu quero criar uma fatura para um cliente que pagou com cheque
[B embaralhada]  26 ['decidiu_certo']
          Quero saber o que esses dados significam.
[B embaralhada]  36 ['decidiu_certo']
          Necessito uma ajuda para or

##### 4. Condition D

In [8]:
D_REGISTRADA = {
    "formatou":           51/51,
    "chamada_utilizavel": 37/37,
    "decidiu_certo":      51/51,
    "json_valido":        36/37,
    "tool_certa":         36/36,
    "params_ok":          36/36,
    "tipos_ok":           36/36,
}

# quebra por tipo, que é o que o experimento D realmente testa
D_POR_TIPO = {"com_tool 4->3": 37/37, "sem_tool 3->4": 14/14}

print("D — contagem invertida (registrada, não recalculada)")
for k, v in D_POR_TIPO.items():
    print(f"  decidiu_certo {k}: {v:.0%}")

D — contagem invertida (registrada, não recalculada)
  decidiu_certo com_tool 4->3: 100%
  decidiu_certo sem_tool 3->4: 100%


##### 5. Final tables

Two new metrics:

- **`formatou`**: produced a usable tag (neither `sem_tag` nor `ambos`).
- **`chamada_utilizavel`**: among the 37 cases that **required** a call, how many produced
  one with a closed tag and valid JSON. **This is the number that describes the goal of the
  project.**

They exist because `decidiu_certo` was secretly **two tests multiplied**: it could only be
computed if there was a tag to extract, so it measured "formatted?" × "decided right?".
Reported as a single number, it hid both: 57% × 93% = 53%.

In [9]:
PARSEAVEL = {"tool_call", "final_answer"}


def metricas(res):
    fmt     = [r for r in res if r["obtido"] in PARSEAVEL]
    alvo    = [r for ex, r in zip(testes, res) if len(ex["messages"]) == 5]
    usaveis = [r for r in alvo if r["obtido"] == "tool_call" and r["json_valido"]]

    def taxa(campo):
        v = [r[campo] for r in res if r[campo] is not None]
        return sum(v) / len(v) if v else None

    return {
        "formatou":           len(fmt) / len(res),
        "chamada_utilizavel": len(usaveis) / len(alvo),
        "decidiu_certo":      taxa("decidiu_certo"),
        "json_valido":        taxa("json_valido"),
        "tool_certa":         taxa("tool_certa"),
        "params_ok":          taxa("params_ok"),
        "tipos_ok":           taxa("tipos_ok"),
    }


COLUNAS = ["formatou", "chamada_utilizavel", "decidiu_certo",
           "json_valido", "tool_certa", "params_ok", "tipos_ok"]

TABELA = {nome: metricas(res) for nome, res in resultados.items()}
TABELA["D contagem invertida"] = D_REGISTRADA

ORDEM = ["A original", "B embaralhada", "D contagem invertida", "C base"]

print(f"{'condicao':22}" + "".join(f"{c[:11]:>13}" for c in COLUNAS))
for nome in ORDEM:
    m = TABELA[nome]
    print(f"{nome:22}" + "".join(
        f"{m[c]:>12.0%} " if m.get(c) is not None else f"{'—':>13}" for c in COLUNAS))

condicao                   formatou  chamada_uti  decidiu_cer  json_valido   tool_certa    params_ok     tipos_ok
A original                    100%         100%         100%         100%         100%         100%         100% 
B embaralhada                  96%          92%          96%          97%         100%         100%         100% 
D contagem invertida          100%         100%         100%          97%         100%         100%         100% 
C base                         57%          46%          53%         100%         100%         100%         100% 


In [10]:
# a leitura em três níveis, do específico ao geral
ft, base = TABELA["A original"], TABELA["C base"]

print(f"{'':52}{'base':>8}{'ajustado':>12}")
print(f"{'produziu chamada utilizável quando devia?':52}"
      f"{base['chamada_utilizavel']:>8.0%}{ft['chamada_utilizavel']:>12.0%}")
print(f"{'entregou resposta utilizável e correta?':52}"
      f"{base['decidiu_certo']:>8.0%}{ft['decidiu_certo']:>12.0%}")

cond = lambda res: (
    sum(r["decidiu_certo"] for r in res if r["obtido"] in PARSEAVEL)
    / sum(1 for r in res if r["obtido"] in PARSEAVEL))
print(f"{'decidiu certo, entre as vezes em que formatou':52}"
      f"{cond(resultados['C base']):>8.0%}{cond(resultados['A original']):>12.0%}")

print("\nRessalva: os denominadores de C são amostra selecionada pelo sucesso no passo")
print("anterior — plausivelmente os casos mais fáceis. Os 93% são estimativa otimista.")

                                                        base    ajustado
produziu chamada utilizável quando devia?                46%        100%
entregou resposta utilizável e correta?                  53%        100%
decidiu certo, entre as vezes em que formatou            93%        100%

Ressalva: os denominadores de C são amostra selecionada pelo sucesso no passo
anterior — plausivelmente os casos mais fáceis. Os 93% são estimativa otimista.


##### 6. Weights & Biases

The original training run used `report_to="none"`, so nothing was sent live; the curve
below is filled in retroactively from the loss table.

In [11]:
import wandb

run = wandb.init(
    entity  = "annajuliaasfiag",
    project = "tool-use-ptbr",
    name    = "gemma4-E2B-lora-r8-3ep",
    notes   = "Run #1. Dataset sintético em português, 30 tools fictícias. "
              "Curva preenchida retroativamente (o treino rodou com report_to='none').",
    config  = {
        "modelo_base": "unsloth/gemma-4-E2B-it",
        "metodo": "QLoRA 4-bit",
        "r": 8, "lora_alpha": 8, "lora_dropout": 0,
        "epocas": 3, "learning_rate": 2e-4,
        "batch": 1, "grad_accum": 8, "batch_efetivo": 8,
        "optim": "adamw_8bit", "max_seq_length": 2048,
        "train_on_responses_only": True,
        "n_treino": 386, "n_validacao": 47, "n_teste": 51,
        "gpu": "Tesla T4", "dtype": "float32",
        "params_treinaveis": 12_668_928,
    },
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/anna-julia/.netrc.
wandb: Currently logged in as: ajuliakj (annajuliaasfiag) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# (passo, perda de treino, perda de validação) do output do trainer no Colab
HISTORICO = [
    (10, 0.149792, 0.939093), (20, 0.089045, 0.646226), (30, 0.077280, 0.593627),
    (40, 0.081639, 0.545749), (50, 0.115101, 0.528203), (60, 0.058817, 0.523526),
    (70, 0.060510, 0.513548), (80, 0.074269, 0.504501), (90, 0.065765, 0.497667),
    (100, 0.071903, 0.492359), (110, 0.056176, 0.493419), (120, 0.060621, 0.491572),
    (130, 0.058784, 0.489435), (140, 0.051503, 0.488113), (147, 0.052627, 0.487914),
]


for passo, treino, validacao in HISTORICO:
    wandb.log({"train/loss": treino, "eval/loss": validacao,
               "epoca": passo / 49}, step=passo)

print(f"{len(HISTORICO)} pontos enviados")

15 pontos enviados


In [ ]:
tabela_wb = wandb.Table(columns=["condicao"] + COLUNAS)
for nome in ORDEM:
    m = TABELA[nome]
    tabela_wb.add_data(nome, *[m.get(c) for c in COLUNAS])
wandb.log({"avaliacao/condicoes": tabela_wb})


wandb.summary["eval_loss_final"]         = HISTORICO[-1][2]
wandb.summary["chamada_utilizavel_ft"]   = ft["chamada_utilizavel"]
wandb.summary["chamada_utilizavel_base"] = base["chamada_utilizavel"]
wandb.summary["formatou_ft"]             = ft["formatou"]
wandb.summary["formatou_base"]           = base["formatou"]
wandb.summary["vazamento_posicao"]       = "refutado"
wandb.summary["vazamento_contagem"]      = "refutado"

wandb.finish()

epoca,▁▂▂▃▃▄▄▅▅▆▆▇▇██
eval/loss,█▃▃▂▂▂▁▁▁▁▁▁▁▁▁
train/loss,█▄▃▃▆▂▂▃▂▂▁▂▂▁▁
chamada_utilizavel_base,0.45946
chamada_utilizavel_ft,1
epoca,3
eval/loss,0.48791
eval_loss_final,0.48791
formatou_base,0.56863
formatou_ft,1
train/loss,0.05263


In [12]:
import wandb

v = wandb.Api().viewer
for attr in ["username", "name", "entity", "teams"]:
    print(f"{attr:10} = {getattr(v, attr, '(não existe)')}")



wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/anna-julia/.netrc.


username   = annajuliaasf
name       = Anna Júlia Ferreira
entity     = models
teams      = []
